# T5, BART -- Encoder Decoder Models

Encoder understand. Decoders generate. Put them back together and you get a model built for input --> output tasks: translate, summarize, rewrite, transcrible.

## Problem definition.

Decoder-only GPT and encoder-only BERT each strip down the 2017 architecture for a different goal. But many tasks are naturally input-output:
* Translation
* Summarization
* Speech recognition
* Structured extraction

The encoder produces a dense representation of the source. The decoder generates the output, cross-attending to that representation at every step.

Two papers defined the modern playbook:

### T5 - Text To Text Transfer Transformer

Pretrained on masked span prediction (corrups spans in the input, decode them in the output)

### BART - Bidirectional and Auto-Regressive Transformer

Denoising autoencoder: corrupt input in multiple ways (shuffle, mask, delete, rotate), ask the decoder to reconstruct the original.


## Basic Concept

Encoder-decoder with cross attention

### The forward loop
```
source tokens --> encoder --> (N_src, d_model) --|
                                                 |
target tokens --> decoder                        |
                   |--> masked self-attention    |
                   |--> cross- attention <-------|
                   |--> FFM
                   |
                  next-token logits
```

### T5 pretraining - span corruption

Pick random spans of input (average length 3 tokens, 15% total). Replace each span with a unique sentinel: <extra_id_0>, ... etx. The decoder outputs only the corrupted spans with their sentiel prefix.

output the map {extra_id_i : token sequence}

```
source: The quick <extra_id_0> fox jumps <extra_id_1> dog
target: <extra_id_0> brown <extra_id_1> over the lazy
```

### BART pretraining - multi-nois denoising

BART tries five noising functions.

1. Token masking
2. Token deletion
3. Text infilling (mask a span, decoder inserts the right length)
4. Sentence permutation
5. Document rotation

### When to pick each variant in 2026

| Task | Encoder-decoder? | Why |
|------|------------------|-----|
| Translation | Yes, usually | Clear source sequence; fixed output distribution; beam search works |
| Speech-to-text | Yes (Whisper) | Input modality differs from output; encoder shapes audio features |
| Chat / reasoning | No, decoder-only | No persistent "input" — the conversation is the sequence |
| Code completion | Usually no | Decoder-only with long context wins; code models like Qwen 2.5 Coder are decoder-only |
| Summarization | Either works | BART, PEGASUS beat earlier decoder-only baselines; modern decoder-only LLMs match them |
| Structured extraction | Either | T5 is clean because "text → text" absorbs any output format |

The trend since ~2022: decoder-only takes over tasks that encoder-decoder used to own because (a) instruction-tuned decoder-only LLMs generalize to anything via prompting, (b) one architecture scales easier than two, (c) RLHF assumes a decoder. Encoder-decoder holds on where input modality differs (speech, images) or where beam search quality matters.

# Build your Own

In [ ]:
import random

def sentinel(i):
    return f"<extra_id_{i}>"

def corrupt_span(tokens, mask_rate=0.15, mean_span=3.0, rng=None):
    if rng is None:
        rng = random.Random()
    n = len(tokens)
    n_mask = max(1, int(round(n * mask_rate)))
    n_spans = max(1, int(round(n_mask / mean_span)))

    positions = list(range(n))
    rng.shuffle(positions)

    starts = []
    used = [False] * n
    span_lengths = []
    remaining = n_mask
    for _ in range(n_spans):
        if remaining <= 0:
            break

        random_order = list(range(n))
        rng.shuffle(random_order)
        chosen_start = None
        for start in random_order:
            if used[start]:
                continue

            length = max(1, int(rng.gauss(mean_span, 1.0)))
            length = min(length, remaining, n - start)
            if length < 1:
                continue
            if any(used[i] for i in range(start, start + length)):
                continue

            chosen_start = start
            for i in range(start, start + length):
                used[i] = True
            starts.append(start)
            span_lengths.append(length)
            remaining -= length
            break
        if chosen_start is None:
            break

    ordered = sorted(zip(starts, span_lengths), key=lambda x: x[0])

    source = []
    target = []
    prev_end = 0
    for idx, (start, length) in enumerate(ordered):
        source.extend(tokens[prev_end: start])
        source.append(sentinel(idx))
        target.append(sentinel(idx))
        target.extend(tokens[start: start + length])
        prev_end = start + length

    source.extend(tokens[prev_end:]) 
    target.append(sentinel(len(ordered)))  # End sentinel
    return source, target

import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

with SectionPrinter("T5 Span Corruption"):
    FAKE_TEXT = "The quick brown fox jumps over the lazy dog"
    def tokenize(text):
        return text.split()
    tokens = tokenize(FAKE_TEXT)

    source, target = corrupt_span(tokens)
    print(source)
    print(target)




=====================T5 Span Corruption=====================
['The', 'quick', 'brown', 'fox', '<extra_id_0>', 'over', 'the', 'lazy', 'dog']
['<extra_id_0>', 'jumps', '<extra_id_1>']


In [ ]:
def token_mask(tokens, mask_rate=0.15, rng=None, mask_token="<mask>"):
    if rng is None:
        rng = random.Random()
    return [mask_token if rng.random() < mask_rate else t for t in tokens]

def token_delete(tokens, delete_rate=0.15, rng=None):
    if rng is None:
        rng = random.Random()
    return [t for t in tokens if rng.random() >= delete_rate]

def token_infill(tokens, rate=0.15, mean_span=3.0, rng=None, mask_token="<mask>"):
    if rng is None:
        rng = random.Random()
    out = []
    i = 0
    n = len(tokens)
    budget = int(n * rate)
    while i < n:
        if budget > 0 and rng.random() < 0.3:
            span_len = max(1, min(int(rng.gauss(mean_span, 1.0)), n - i, budget))
            out.append(mask_token)
            budget -= span_len
            i += span_len
        else:
            out.append(tokens[i])
            i += 1
    return out

def sentence_permute(sentences, rng=None):
    if rng is None:
        rng = random.Random()
    sents = list(sentences)
    rng.shuffle(sents)
    return sents

def document_rotate(tokens, rng=None):
    if rng is None:
        rng = random.Random()
    if len(tokens) <=1:
        return tokens
    pivot = rng.randrange(1, len(tokens))

    return tokens[pivot:] + tokens[:pivot]

with SectionPrinter("BART Denoising"):
    FAKE_TEXT = "The quick brown fox jumps over the lazy dog, the quick brown fox jumps over the lazy dog"
    tokens = tokenize(FAKE_TEXT)

    print(token_mask(tokens))
    print(token_delete(tokens))
    print(token_infill(tokens))

    combinations = [
        token_mask,
        token_delete,
        token_infill,
        sentence_permute,
        document_rotate
    ]

    tk = tokens
    for comb in combinations:
        tk = comb(tk)
        print(tk)


=======================BART Denoising=======================
['The', 'quick', 'brown', 'fox', '<mask>', 'over', 'the', 'lazy', '<mask>', 'the', '<mask>', 'brown', 'fox', 'jumps', 'over', 'the', '<mask>', 'dog']
['The', 'quick', 'brown', 'jumps', 'over', 'the', 'lazy', 'dog,', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
['<mask>', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog,', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
